# 03 — Prognostic Workflow

This notebook demonstrates the **prognostic** use case of the ISA-PHM wrapper:
analysing a bearing that was run to failure and extracting meaningful health
indicators that degrade over time.

**Dataset:** XJTU-SY Bearing Run-to-Failure Dataset  
**Reference:** Lei et al., *Machinery Health Prognostics*, 2016  
_15 rolling-element bearings — 3 operating conditions × 5 bearings each_

| Condition | Motor speed | Radial load |
|-----------|-------------|-------------|
| 35 Hz     | ~2100 RPM   | 12 kN       |
| 37.5 Hz   | ~2250 RPM   | 11 kN       |
| 40 Hz     | ~2400 RPM   | 10 kN       |

Each bearing has **100 – 2500 runs** of 32 768 samples each (at 25.6 kHz, ~1.28 s per snapshot), recorded every minute until failure.

---

**What you will learn:**
1. Load a multi-run prognostic dataset via `ISAWrapper`
2. Inspect the investigation and study metadata
3. Compute lifecycle features across all runs with parallel loading
4. Plot a single-bearing degradation trajectory
5. Cross-bearing comparison with `plot_multi_lifecycle()`
6. Export a labeled dataset for RUL (remaining useful life) modelling
7. Time-domain and FFT analysis of early vs late runs

In [1]:
%pip install -q pydantic pandas numpy scipy bokeh --quiet

Note: you may need to restart the kernel to use updated packages.


In [2]:
import warnings
warnings.filterwarnings('ignore')          # suppress duplicate-@id notices from this dataset

import logging
logging.getLogger('isa_phm').setLevel(logging.ERROR)

import sys
from pathlib import Path

# ── Add wrapper to path (when running without pip install) ─────────────────
WRAPPER_ROOT = Path().resolve().parent
if str(WRAPPER_ROOT) not in sys.path:
    sys.path.insert(0, str(WRAPPER_ROOT))

from isa_phm import ISAWrapper

from bokeh.io import output_notebook
from bokeh.plotting import show as bokeh_show
output_notebook()

# ── Paths ──────────────────────────────────────────────────────────────────
# Adjust ISA_JSON to the location on your machine.
ISA_JSON = Path(r"G:\XJTU-SY_Bearing_Datasets\XJTU-SY Bearing Datasets-ISA-PHM.json")

print(f"ISA-JSON : {ISA_JSON}")
print(f"Exists   : {ISA_JSON.exists()}")

Loading BokehJS ...

ISA-JSON : G:\XJTU-SY_Bearing_Datasets\XJTU-SY Bearing Datasets-ISA-PHM.json
Exists   : True


## 1. Load the wrapper

In [3]:
wrapper = ISAWrapper(
    path=ISA_JSON,
    strict_validation=False,   # isatools not needed for prognostic datasets
    cache_maxsize=10,          # keep memory usage low for large run-to-failure datasets
)

overview = wrapper.investigation_overview()
print(overview)
print(f"\n{overview.n_studies} studies loaded")

title='XJTU-SY Bearing Datasets' description='XJTU-SY bearing datasets are provided by the Institute of Design Science and Basic Component at Xi’an Jiaotong University (XJTU), Shaanxi, P.R. China (http://gr.xjtu.edu.cn/web/yaguolei) and the Changxing Sumyoung Technology Co., Ltd. (SY), Zhejiang, P.R. China (https://www.sumyoungtech.com.cn). The datasets contain complete run-to-failure data of 15 rolling element bearings that were acquired by conducting many accelerated degradation experiments. These datasets are publicly available and anyone can use them to validate prognostics algorithms of rolling element bearings.' identifier='694f7a38-1cb8-46b0-aba1-8769e5ddf753' experiment_type='prognostics-experiment' n_studies=15 n_contacts=4 studies=[StudySummary(study_id='12a95728-7b62-4f7f-8c49-82b4512fe5d2', title='Bearing 1_1', n_assays=2, n_runs=123, n_factors=5), StudySummary(study_id='f8dd58d5-e816-4da4-9b76-f4c12548912c', title='Bearing 1_2', n_assays=2, n_runs=161, n_factors=5), StudyS

## 2. Study list — all 15 bearings

In [4]:
import pandas as pd

studies_df = pd.DataFrame([
    {"title": s.title, "n_assays": s.n_assays, "n_runs": s.n_runs, "n_factors": s.n_factors}
    for s in wrapper.list_studies()
])
print(f"All {len(studies_df)} bearings in the dataset:")
display(studies_df)

All 15 bearings in the dataset:


,title,n_assays,n_runs,n_factors
0,Bearing 1_1,2,123,5
1,Bearing 1_2,2,161,5
2,Bearing 1_3,2,158,5
3,Bearing 1_4,2,122,5
4,Bearing 1_5,2,52,5
5,Bearing 2_1,2,491,5
6,Bearing 2_2,2,161,5
7,Bearing 2_3,2,533,5
8,Bearing 2_4,2,42,5
9,Bearing 2_5,2,339,5


## 3. Navigate into one study — Bearing 1_1

Each bearing is one ISA-PHM **study**.  We will start with `Bearing 1_1`
(35 Hz / 12 kN operating condition, outer-race fault type).
It ran for **123 minutes** until failure.

In [5]:
STUDY_TITLE = "Bearing 1_1"

study = wrapper.study(STUDY_TITLE)

print(f"Study          : {study.title}")
print(f"Run count      : {study.run_count}  (one snapshot per minute)")
print(f"Sensor channels: {len(study.list_assays())}")

# Show the sensor catalog
catalog = study.sensor_catalog()
display(catalog)

Study          : Bearing 1_1
Run count      : 123  (one snapshot per minute)
Sensor channels: 2


,assay_id,sensor_alias,measurement_type,technology_type,technology_platform,n_runs,n_raw_files,n_processed_files,fs_hz,unit
0,a_st01_se01,a_st01_se01,Vibration,Accelerometer,PCB 352C33,123,123,0,25600.0,min
1,a_st01_se02,a_st01_se02,Vibration,Accelerometer,PCB 352C33,123,123,0,25600.0,min


## 4. Experimental conditions (factor values)

In [6]:
# Full test matrix — rows are factors, columns are conditions
print("Test matrix (factor × condition):")
display(study.test_matrix())

# Condition breakdown
print("\nOperating conditions:")
display(study.operating_conditions())

print("\nFault conditions:")
display(study.fault_conditions())

Test matrix (factor × condition):


,variable,type,unit,Condition 1,Condition 2,Condition 3,Condition 4,Condition 5,Condition 6,Condition 7,...,Condition 114,Condition 115,Condition 116,Condition 117,Condition 118,Condition 119,Condition 120,Condition 121,Condition 122,Condition 123
0,Fault Type,Qualitative fault specification,,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,...,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race
1,Bearing Lifetime,Quantitative fault specification,min,123 min,122 min,121 min,120 min,119 min,118 min,117 min,...,10 min,9 min,8 min,7 min,6 min,5 min,4 min,3 min,2 min,1 min
2,Motor speed,Operating condition,RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,...,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM
3,Pressure Axial,Operating condition,kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,...,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN
4,Pressure Radial,Operating condition,kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,...,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN



Operating conditions:


,variable,type,unit,Condition 1,Condition 2,Condition 3,Condition 4,Condition 5,Condition 6,Condition 7,...,Condition 114,Condition 115,Condition 116,Condition 117,Condition 118,Condition 119,Condition 120,Condition 121,Condition 122,Condition 123
0,Motor speed,Operating condition,RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,...,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM,2100 RPM
1,Pressure Axial,Operating condition,kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,...,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN
2,Pressure Radial,Operating condition,kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,...,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN,12 kN



Fault conditions:


,variable,type,unit,Condition 1,Condition 2,Condition 3,Condition 4,Condition 5,Condition 6,Condition 7,...,Condition 114,Condition 115,Condition 116,Condition 117,Condition 118,Condition 119,Condition 120,Condition 121,Condition 122,Condition 123
0,Fault Type,Qualitative fault specification,,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,...,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race,Outer Race
1,Bearing Lifetime,Quantitative fault specification,min,123 min,122 min,121 min,120 min,119 min,118 min,117 min,...,10 min,9 min,8 min,7 min,6 min,5 min,4 min,3 min,2 min,1 min


## 5.  Lifecycle feature extraction (parallel)

`assay.lifecycle_features()` computes 8 scalar statistics for **every** run  
using a thread-pool — it never loads more than `n_workers` CSVs simultaneously.

Computed features:

| Feature | Description |
|---------|-------------|
| `rms` | Root-mean-square amplitude — most sensitive to bearing fatigue |
| `kurtosis` | Impulsiveness — spikes sharply near failure |
| `crest_factor` | Peak / RMS ratio — amplifies early micro-pitting signals |
| `peak2peak` | Dynamic range (max − min) |
| `std` | Standard deviation |
| `skewness` | Signal asymmetry |
| `mean` | DC offset |
| `max` | Absolute maximum |

In [7]:
ASSAY_ID = "a_st01_se01"   # horizontal accelerometer

assay = study.assay(ASSAY_ID)
print(f"Assay  : {assay.assay_id}  ({assay.run_count} runs)")

# Parallel load — uses a thread pool (n_workers=None → auto)
lc = assay.lifecycle_features(file_type="raw", n_workers=8)

print(f"\nLifecycle DataFrame shape : {lc.shape}")
print(f"Columns                   : {list(lc.columns)}")
display(lc[["run_number", "rms", "kurtosis", "crest_factor", "peak2peak"]].head(5))

Assay  : a_st01_se01  (123 runs)

Lifecycle DataFrame shape : (123, 17)
Columns                   : ['run_id', 'run_number', 'study_id', 'assay_id', 'rms', 'max', 'mean', 'peak2peak', 'kurtosis', 'std', 'crest_factor', 'skewness', 'fv_Fault Type', 'fv_Bearing Lifetime', 'fv_Motor speed', 'fv_Pressure Axial', 'fv_Pressure Radial']


,run_number,rms,kurtosis,crest_factor,peak2peak
0,1,0.563890,0.071352,4.486485,4.884219
1,2,0.589078,0.137942,6.150471,6.818748
2,3,0.589536,0.246259,5.634563,6.517446
3,4,0.597274,0.248190,4.809554,5.412161
4,5,0.604645,0.393918,6.841896,7.663488


## 6. Plot degradation trajectory — single bearing

RMS amplitude is the most reliable early-degradation indicator for rolling-element bearings.
The lifecycle curve shows a clear run-up phase, a stable operating period,
and a sharp rise towards failure.

In [8]:
fig_rms = assay.plot_lifecycle(feature="rms", file_type="raw")
bokeh_show(fig_rms)

In [9]:
# Kurtosis rises much earlier than RMS — useful for early-fault detection
fig_kurt = assay.plot_lifecycle(feature="kurtosis", file_type="raw")
bokeh_show(fig_kurt)

## 7. Cross-bearing comparison with `plot_multi_lifecycle()`

Overlay the RMS lifecycle curves for **all five bearings** in the 35 Hz / 12 kN
condition on a single interactive figure.  Click a legend entry to hide/show
individual bearing traces.

In [25]:
# Load lifecycle features for all 35 Hz / 12 kN bearings.
# Each study has its own assay IDs (a_st01_se01, a_st02_se01, …),
# so we take the first assay from each study rather than hardcoding the ID.
lc_dict: dict[str, pd.DataFrame] = {}

for bearing_title in ["Bearing 2_1", "Bearing 2_2", "Bearing 2_3", "Bearing 2_4", "Bearing 2_5"]:
    s = wrapper.study(bearing_title)
    first_assay_id = s.list_assays()[1].assay_id   # horizontal channel (se01)
    a = s.assay(first_assay_id)
    lc_b = a.lifecycle_features(file_type="raw", n_workers=8)
    lc_dict[bearing_title] = lc_b
    print(f"  {bearing_title} ({first_assay_id}): {len(lc_b)} runs loaded")

print(f"\n{len(lc_dict)} bearings ready for comparison")

  Bearing 2_1 (a_st06_se02): 491 runs loaded
  Bearing 2_2 (a_st07_se02): 161 runs loaded
  Bearing 2_3 (a_st08_se02): 533 runs loaded
  Bearing 2_4 (a_st09_se02): 42 runs loaded
  Bearing 2_5 (a_st10_se02): 339 runs loaded

5 bearings ready for comparison


In [26]:
from isa_phm.plotter import ISAPlotter

plotter = ISAPlotter()

fig_multi = plotter.plot_multi_lifecycle(
    lc_dict,
    feature="rms",
    title="40 Hz / 12 kN — Bearing RMS Degradation (Vertical Channel)",
)
bokeh_show(fig_multi)

In [12]:
# Crest factor comparison — often the earliest-rising indicator
fig_crest = plotter.plot_multi_lifecycle(
    lc_dict,
    feature="crest_factor",
    title="35 Hz / 12 kN — Crest Factor Degradation",
)
bokeh_show(fig_crest)

## 8. Time-domain waveform — early vs end of life

Comparing the raw vibration waveform at run 1 (healthy) against the last run  
(near failure) reveals the fault-induced impulsive nature of the signal.

In [13]:
# study and assay are already set from Section 5 (Bearing 1_1, horizontal channel)
study_11 = wrapper.study("Bearing 1_1")
assay_11 = study_11.assay(study_11.list_assays()[0].assay_id)

# First run (healthy baseline)
first_run = assay_11.list_runs()[0]
# Last run (near failure)
last_run  = assay_11.list_runs()[-1]

print(f"First run : {first_run.run_id}  (run #{first_run.run_number})")
print(f"Last run  : {last_run.run_id}  (run #{last_run.run_number})")

First run : run_001  (run #1)
Last run  : run_123  (run #123)


In [14]:
# Healthy — run 1
fig_early = assay_11.plot_timeseries(run_id=first_run.run_id, file_type="raw")
bokeh_show(fig_early)

In [ ]:
# Near failure — last run
fig_late = assay_11.plot_timeseries(run_id=last_run.run_id, file_type="raw")
bokeh_show(fig_late)

## 9. FFT spectrum — healthy vs degraded

The bearing fault frequency (BPFO) and its harmonics are clearly visible
in the FFT of the degraded signal but absent in the healthy baseline.

In [16]:
# Load both signals
df_early = assay_11.load_dataframe(run_id=first_run.run_id, file_type="raw")
df_late  = assay_11.load_dataframe(run_id=last_run.run_id,  file_type="raw")

FS = 25_600.0  # Hz — declared in ISA-JSON protocol parameters

fig_fft_early = assay_11.plot_frequency_domain(df=df_early, fs=FS, log_scale=True)
bokeh_show(fig_fft_early)

In [17]:
fig_fft_late = assay_11.plot_frequency_domain(df=df_late, fs=FS, log_scale=True)
bokeh_show(fig_fft_late)

## 10. Feature correlation across runs

The correlation heatmap shows how the 8 scalar features co-evolve across all
runs.  RMS, std, peak2peak, and max typically form a tight cluster.
Kurtosis and crest_factor are often anti-correlated with RMS at early stages
(high impulsiveness before amplitude rises) — this is the classic
"kurtosis reversal" effect.

In [18]:
# Use the lc DataFrame already computed in section 5
fig_corr = assay_11.plot_correlation(file_type="raw")
bokeh_show(fig_corr)

## 11. Amplitude variability across all runs

The variability boxplot shows the distribution of raw signal amplitude
at each captured run.  The growing spread towards the end reflects
the impulse trains caused by the bearing defect.

In [20]:
# Show variability for a subset of runs to keep rendering fast
all_runs = assay_11.list_runs()
# Subsample: first 10 + last 10 runs
subset_run_ids = (
    [r.run_id for r in all_runs[:10]] +
    [r.run_id for r in all_runs[-10:]]
)

fig_var = assay_11.plot_variability(run_ids=[r.run_id for r in all_runs], file_type="raw")
bokeh_show(fig_var)

## 12. ML-ready labeled export with RUL column

`study.export_labeled_dataset()` produces a tidy DataFrame with one row per
sensor sample, annotated with all ISA-PHM factor values.

For RUL modelling we add a **Remaining Useful Life** column computed from the
`Bearing Lifetime` factor, which holds the remaining lifetime in minutes at the
time each run was recorded.

In [22]:
# Re-use lifecycle features computed in section 5 for Bearing 1_1 horizontal
lc_11 = assay_11.lifecycle_features(file_type="raw")

# Get fault labels (includes Bearing Lifetime factor)
labels_11 = study_11.get_fault_labels()

print("Fault labels head:")
display(labels_11.head())

# In this dataset 'Bearing Lifetime' is the REMAINING time (minutes) at each run.
if 'Bearing Lifetime' in labels_11.columns:
    labels_11['rul_min'] = pd.to_numeric(labels_11['Bearing Lifetime'], errors='coerce')

training_df = lc_11.merge(labels_11[['assay_id','run_id','rul_min']], on=['assay_id','run_id'])

print(f"\nTraining DataFrame shape : {training_df.shape}")
print(f"Columns                  : {list(training_df.columns)}")
display(training_df[['run_number','rms','kurtosis','crest_factor','rul_min']].head(5))

Fault labels head:


,assay_id,run_id,run_number,Fault Type,Bearing Lifetime
0,a_st01_se01,run_001,1,Outer Race,123 min
1,a_st01_se01,run_002,2,Outer Race,122 min
2,a_st01_se01,run_003,3,Outer Race,121 min
3,a_st01_se01,run_004,4,Outer Race,120 min
4,a_st01_se01,run_005,5,Outer Race,119 min



Training DataFrame shape : (123, 18)
Columns                  : ['run_id', 'run_number', 'study_id', 'assay_id', 'rms', 'max', 'mean', 'peak2peak', 'kurtosis', 'std', 'crest_factor', 'skewness', 'fv_Fault Type', 'fv_Bearing Lifetime', 'fv_Motor speed', 'fv_Pressure Axial', 'fv_Pressure Radial', 'rul_min']


,run_number,rms,kurtosis,crest_factor,rul_min
0,1,0.563890,0.071352,4.486485,NaN
1,2,0.589078,0.137942,6.150471,NaN
2,3,0.589536,0.246259,5.634563,NaN
3,4,0.597274,0.248190,4.809554,NaN
4,5,0.604645,0.393918,6.841896,NaN


## 13. Cross-condition comparison — all three operating conditions

Bearing 3_1 has 2538 runs (40 Hz / 10 kN) — the longest degradation trajectory
in the dataset.  Comparing it with Bearing 1_1 (35 Hz / 12 kN) shows how
operating conditions affect degradation speed.

In [21]:
# Note: this loads a 2538-run dataset — may take 10-30 s
study_31 = wrapper.study("Bearing 3_1")
assay_31 = study_31.assay(study_31.list_assays()[0].assay_id)
print(f"Bearing 3_1 run count: {assay_31.run_count}")

lc_31 = assay_31.lifecycle_features(file_type="raw", n_workers=8)
print(f"Lifecycle features shape: {lc_31.shape}")

Bearing 3_1 run count: 2538
Lifecycle features shape: (2538, 17)


In [22]:
# Overlay Bearing 1_1 and Bearing 3_1 on one plot
cross_condition_dict = {
    "Bearing 1_1 (35Hz/12kN, 123 runs)": lc_dict["Bearing 1_1"],
    "Bearing 3_1 (40Hz/10kN, 2538 runs)": lc_31,
}

fig_cross = plotter.plot_multi_lifecycle(
    cross_condition_dict,
    feature="rms",
    title="Cross-condition RMS Degradation Comparison",
)
bokeh_show(fig_cross)

---

## What's next?

| Notebook | Topic |
|---|---|
| `01_getting_started.ipynb` | API tour  |
| `02_diagnostic_workflow.ipynb` | Single-run diagnostics, fault classification, labeled export |

**Ideas for further exploration:**
- Fit a simple RUL regressor (e.g. linear regression or random forest) on the `training_df` from Section 12
- Compare RMS vs kurtosis as a health indicator using `plot_multi_lifecycle()`
- Apply `study.load_multi_sensor_dataframe()` to combine the horizontal and vertical accelerometers for 2-channel input features
- Export a combined labeled dataset across all 15 bearings for a dataset-level ML experiment